# Lu Broadband Directional Coupler with GDSFactory FDTD

Build a silicon coupler with an asymmetric phase-control section and inspect its two output transmissions.

**You will:** build the component, configure the physical stack and 30 nm grid, preview the domain, and inspect transmission and a field monitor.

**Requirements:** [GDSFactory+](https://gdsfactory.com) cloud simulation access.

**Reference run:** 30 nm cells · 147,849,728 total allocated cells, including PML and padding.

<div class="admonition quote">
<p class="admonition-title">Original paper</p>
<p>Z. Lu et al., “Broadband silicon photonic directional coupler using asymmetric-waveguide based phase control,” Optics Express 23, 3795–3808 (2015). <a href="https://doi.org/10.1364/OE.23.003795">Read the paper · DOI 10.1364/OE.23.003795</a>.</p>
</div>

## 1. Build the component

All geometry below is in micrometers. The model includes fixed 200 nm straight leads at its modal ports.

In [ ]:
import math

import gdsfactory as gf
import matplotlib.pyplot as plt
import numpy as np
from gdsfactory.technology import LayerLevel, LayerStack, LogicalLayer

from gsim import fdtd

gf.gpdk.PDK.activate()


def level(layer, material, zmin, thickness):
    return LayerLevel(
        layer=LogicalLayer(layer=layer),
        material=material,
        zmin=zmin,
        thickness=thickness,
        mesh_order=1,
        sidewall_angle=0,
    )

In [ ]:
def extend_ports(component, length_um):
    """Append straight access guides and move each existing optical port."""
    ports = list(component.ports)
    component.ports.clear()
    for port in ports:
        angle = math.radians(port.orientation)
        normal = np.round([math.cos(angle), math.sin(angle)], 12)
        tangent = np.array([-normal[1], normal[0]])
        start = np.asarray(port.center)
        # GDS polygons are quantized to 1 nm; ports must share that face.
        end = np.round(start + length_um * normal, 3)
        half_width = port.width / 2
        component.add_polygon(
            [
                start - half_width * tangent,
                end - half_width * tangent,
                end + half_width * tangent,
                start + half_width * tangent,
            ],
            layer=port.layer,
        )
        component.add_port(
            name=port.name,
            center=tuple(end),
            width=port.width,
            orientation=port.orientation,
            layer=port.layer,
        )

In [ ]:
component = gf.Component()
reference = component << gf.components.coupler_broadband(
    w_sc=0.5,
    gap_sc=0.2,
    w_top=0.6,
    gap_pc=0.3,
    legnth_taper=1,
    length_coupler_straight=12.4,
    lenght_coupler_big_gap=4.6,
    radius=5,
    bend="bend_circular",
)
component.add_ports(reference.ports)
layers = {"core": level((1, 0), "Si", 0, 0.22)}


extend_ports(component, 0.2)

## 2. Configure the simulation

Use explicit layer thicknesses and frozen source-center indices to reproduce the saved result. The 1.28 µm target PML is rounded up to whole cells. Materials are nondispersive over the sweep.

In [ ]:
pdk = gf.Pdk(name="fdtd_lu_broadband_coupler", layer_stack=LayerStack(layers=layers))
simulation = fdtd.Simulation(pdk=pdk)
simulation.geometry(component)
simulation.materials(
    background="SiO2", overrides={"Si": 3.477723756220899, "SiO2": 1.4440236217032607}
)
simulation.source(port="o1", wavelength_um=1.55, wavelength_span_um=0.1)
simulation.solver(cell_size_nm=30, max_wall_seconds=1800)
simulation.domain(
    pml_cells=43,
    x_bounds=((-21.95), (21.95)),
    y_bounds=((-5.65), (5.45)),
    z_bounds=((-1.0), (1.22)),
)

lower, upper = np.array([[-21.95, -5.65, -1.0], [21.95, 5.45, 1.22]])
simulation.monitors.add_plane(
    "overview_xy",
    center_um=((lower[0] + upper[0]) / 2, (lower[1] + upper[1]) / 2, 0.11),
    size_um=(upper[0] - lower[0] - 0.24, upper[1] - lower[1] - 0.24, 0),
    normal="+z",
    flux=False,
    wavelengths_um=[1.55],
    heatmap=fdtd.Heatmap(quantity="intensity", wavelengths_um=[1.55]),
)

## 3. Inspect the simulation

Use the built-in 2D views to inspect the core, ports, monitor and PML. The top view passes through the core; the side view shows the vertical stack.

In [ ]:
simulation.plot_2d(axis="z", position_um=0.11, figsize=(10, 4), legend=False)
plt.show()

In [ ]:
simulation.plot_2d(axis="y", position_um=0, figsize=(10, 4), legend=False)
plt.show()

Rotate and zoom the built-in 3D viewer to inspect the complete simulation geometry.

In [ ]:
simulation.plot_3d()

## 4. Run the simulation

Reuse a matching result when it is available in the cloud cache.

In [ ]:
result = simulation.run(check_cache=True)

In [ ]:
print(f"Total allocated cells: {result.grid['total_cells']:,}")
print(f"Solver time: {result.timing['run_seconds']:.1f} s")

## 5. Inspect transmission

Plot incident-power-normalized modal transmission. Invalid source-band samples remain masked. Default Matplotlib figure size and fonts are used.

In [ ]:
fig, ax = plt.subplots()
for port in ["o3", "o4"]:
    trace = result.s_parameters[port, "o1"]
    power = np.where(trace.valid, np.abs(trace.values) ** 2, np.nan)
    ax.plot(result.wavelength_um * 1000, power, label=f"|S{port[1:]}1|²")
ax.set(
    xlabel="Wavelength (nm)",
    ylabel="Power transmission",
    title="Lu Broadband Directional Coupler",
)
ax.grid(alpha=0.2)
ax.legend()
plt.show()

## 6. Inspect the top-view field

Overlay the silicon or silicon-nitride layout on the source-center intensity monitor. The x/y axes use equal aspect.

In [ ]:
monitor = result.monitors["overview_xy"]
fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")
monitor.plot_heatmap(
    wavelength_um=simulation.source.wavelength_um,
    quantity="intensity",
    ax=ax,
    cmap="viridis",
)
for points in component.get_polygons_points(by="tuple")[(1, 0)]:
    outline = np.vstack([points, points[0]])
    ax.plot(outline[:, 0], outline[:, 1], color="white", linewidth=0.5)
ax.set_aspect("equal", adjustable="box")
plt.show()

## Model scope

This is a parametric reconstruction with circular access bends and 200 nm port leads, not an author-supplied mask. The paper reports output-normalized splitting, so it does not independently establish absolute throughput. The saved 30 nm spectrum is an example result, not a claim of mesh convergence.

The saved result reached its finite-time stopping condition. That is separate from spatial mesh convergence. The reference campaign enforced a 30-minute wall-time budget.